<a href="https://colab.research.google.com/github/Jaguar838/ml-zoomcamp/blob/master/HW/hw09/09-hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework

In this homework, we'll deploy the Straight vs Curly Hair Type model we trained in the\n[previous homework](../08-deep-learning/homework.md).
Download the model files from here:
 https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data

 https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx

In [ ]:
!wget https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data
!wget https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx

## Question 1
To be able to use this model, we need to know the name of the input and output nodes.
What's the name of the output:
*   `output`
*   `sigmoid`
*   `softmax`
*   `prediction`





In [ ]:
!pip install onnxruntime

In [7]:
import onnxruntime as ort

model_path = 'hair_classifier_v1.onnx'
session = ort.InferenceSession(model_path)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print(f"Input name: {input_name}")
print(f"Output name: {output_name}")

Input name: input
Output name: output


**Answer:** The name of the output is `output`.

## Preparing the image
You'll need some code for downloading and resizing images. You can use this code:
```python\nfrom io import BytesIO\nfrom urllib import request\n\nfrom PIL import Image\n\ndef download_image(url):\n    with request.urlopen(url) as resp:\n        buffer = resp.read()\n    stream = BytesIO(buffer)\n    img = Image.open(stream)\n    return img\n\n\ndef prepare_image(img, target_size):\n    if img.mode != 'RGB':\n        img = img.convert('RGB')\n    img = img.resize(target_size, Image.NEAREST)\n    return img\n```\n\nFor that, you'll need to have `pillow` installed:\n\n```bash\npip install pillow\n```\n\n## Question 2: Target size\n\nLet's download and resize this image: \n\nhttps://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\n\nBased on the previous homework, what should be the target size for the image?\n\n* 64x64\n* 128x128\n* 200x200\n* 256x256

In [8]:
from io import BytesIO
from urllib import request
from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

# From homework 8, the target size is 200x200
target_size = (200, 200)
print(f"The target size is {target_size}")

The target size is (200, 200)


**Answer:** The target size is `200x200`.

## Question 3\n\nNow we need to turn the image into numpy array and pre-process it. \n\n> Tip: Check the previous homework. What was the pre-processing \n> we did there?\n\nAfter the pre-processing, what's the value in the first pixel, the R channel?\n\n* -10.73\n* -1.073\n* 1.073\n* 10.73

In [18]:
import numpy as np

# Assuming download_image and prepare_image are defined in a previous cell
# and target_size is also defined.

def preprocess_image(img):
    x = np.array(img, dtype='float32')
    # Scale to [0, 1]
    x = x / 255.0
    # Normalize with ImageNet mean and std
    mean = np.array([0.485, 0.456, 0.406], dtype='float32')
    std = np.array([0.229, 0.224, 0.225], dtype='float32')
    x = (x - mean) / std
    # Change from (H, W, C) to (C, H, W)
    x = x.transpose(2, 0, 1)
    return x

# These variables should be available from previous cells' execution:
# image_url (e.g., 'https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg')
# target_size (e.g., (200, 200))
# img = download_image(image_url)
# img_prepared = prepare_image(img, target_size)

# To get the answer for Question 3, we need to execute the preprocessing and then inspect the first pixel.
# Assuming 'img_prepared' and 'preprocess_image' are accessible from previous steps.

# Let's re-run the necessary parts to get the preprocessed_image and print the first pixel R channel
from io import BytesIO
from urllib import request
from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

image_url = 'https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg'
target_size = (200, 200)

img = download_image(image_url)
img_prepared = prepare_image(img, target_size)

preprocessed_image_q3 = preprocess_image(img_prepared)

# For Question 3: value in the first pixel, the R channel
first_pixel_r = preprocessed_image_q3[0, 0, 0]
print(f"The value of the R channel of the first pixel (after preprocessing) is: {first_pixel_r:.3f}")

The value of the R channel of the first pixel (after preprocessing) is: -1.073


**Answer:** The value is approximately `-1.073`.

## Question 4\n\nNow let's apply this model to this image. What's the output of the model?\n\n* 0.09\n* 0.49\n* 0.69\n* 0.89

In [20]:
import onnxruntime as ort
import numpy as np
from PIL import Image
from io import BytesIO
from urllib import request
import sys

url ='https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg'

def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

def preprocess_image(img):
    x = np.array(img, dtype='float32')
    x = x / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype='float32')
    std = np.array([0.229, 0.224, 0.225], dtype='float32')
    x = (x - mean) / std
    x = x.transpose(2, 0, 1)
    return x

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def predict(url):
    model_path = './hair_classifier_v1.onnx'
    session = ort.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    img = download_image(url)
    img_prepared = prepare_image(img, (200, 200))
    preprocessed_image = preprocess_image(img_prepared)
    input_tensor = np.expand_dims(preprocessed_image, axis=0)

    result = session.run([output_name], {input_name: input_tensor})[0]
    return result[0][0]

def lambda_handler(event, context):
    url = event['url']
    prediction = predict(url)
    return {
        'prediction': float(prediction)
    }

if __name__ == "__main__":
    # Use the predefined URL for execution in Colab
    prediction = predict(url)
    print(f"{prediction:.3f}")

0.092


**Answer:** The output is

1.   List item
2.   List item

approximately `0.09`.

## Question 5\n\nDownload the base image `agrigorev/model-2025-hairstyle:v1`. You can do it with [`docker pull`](https://docs.docker.com/engine/reference/commandline/pull/).\n\nSo what's the size of this base image?\n\n* 88 Mb\n* 208 Mb\n* 608 Mb\n* 1208 Mb\n\nYou can get this information when running `docker images` - it'll be in the \"SIZE\" column.

In [ ]:
!docker pull agrigorev/model-2025-hairstyle:v1

In [ ]:
!docker images agrigorev/model-2025-hairstyle

**Answer:** The size is `1208 Mb`

## Question 6\n\nNow let's extend this docker image, install all the required libraries\nand add the code for lambda.\n\nYou don't need to include the model in the image. It's already included. \nThe name of the file with the model is `hair_classifier_empty.onnx` and it's \nin the current workdir in the image (see the Dockerfile above for the \nreference). \nThe provided model requires the same preprocessing for images regarding target size and rescaling the value range than used in homework 8.\n\nNow run the container locally.\n\nScore this image: https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\n\nWhat's the output from the model?\n\n* -1.0\n* -0.10\n* 0.10\n* 1.0

Here's the `Dockerfile`:

In [22]:
!cat Dockerfile

FROM agrigorev/model-2025-hairstyle:v1

RUN pip install onnxruntime numpy pillow

COPY lambda_function.py .

CMD [ "lambda_function.lambda_handler" ]

And the `lambda_function.py`:

In [23]:
!cat lambda_function.py

import onnxruntime as ort
import numpy as np
from PIL import Image
from io import BytesIO
from urllib import request
import sys

def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

def preprocess_image(img):
    x = np.array(img, dtype='float32')
    x = x / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype='float32')
    std = np.array([0.229, 0.224, 0.225], dtype='float32')
    x = (x - mean) / std
    x = x.transpose(2, 0, 1)
    return x

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def predict(url):
    model_path = 'hair_classifier_empty.onnx'
    session = ort.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    img = download_image(url)
    img_prepared = prepare_image(img

Let's build the image.

In [ ]:
!docker build -t homework-image .

And now let's run it and get the prediction.

In [ ]:
!docker run -it --rm homework-image '{\"url\": \"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\"}'

**Answer:** The output from the model is `-0.10`.